# Explore videos and create a complementary-half experiment

Connect → inspect metadata → create or reopen the 5-subject rehearsal → verify → optionally download.
Run cells in order against the matching `/api/v1` website. No presentation framework is needed.


In [ ]:
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import display
from dbp_api import Client, Experiment, Assignments, Session, Trial, MetricFilter, Video
# from dbp_api import Image  # Future modality; not yet supported by the server.

website_url = input("Website URL [http://127.0.0.1:8773]: ").strip() or "http://127.0.0.1:8773"
client = Client(website_url, timeout=120)
client.login(input("Username: "), getpass("Password: "))
print("Connected")


## All metrics and coverage

`known` counts videos with a value, including zero or false. `unknown` means no
usable value. Each metric has its own coverage; adding these counts would double
count videos. The request returns aggregates for the full dataset, but only one
video row. No duration or content filter is applied here.


In [ ]:
inventory = client.metrics()
baseline = client.query_media(limit=1)
print(f"{baseline['matching']:,} of {baseline['total']:,} videos")

def metric_coverage(result):
    records = []
    for metric in inventory["metrics"]:
        summary = result["distributions"].get(metric["id"])
        known = summary["known"] if summary is not None else None
        records.append({
            "Metric": metric["label"],
            "ID": metric["id"],
            "Type": metric["kind"],
            "Unit": metric["unit"],
            "Operators": ", ".join(metric["operators"]),
            "Measured videos": known,
            "Unknown videos": summary["unknown"] if summary is not None else None,
            "Measured %": 100 * known / result["matching"] if known is not None and result["matching"] else None,
        })
    return pd.DataFrame(records)

with pd.option_context("display.max_rows", None):
    display(metric_coverage(baseline))


## Filter videos

Edit the content query and metric filters below. All constraints are combined.
Content search uses visual captions and spoken transcripts (`corpus="both"`);
choose `captions` or `transcripts` to restrict the source. An empty query matches
all content. Missing metric values do not satisfy numeric comparisons.


In [ ]:
content_query = ""
filters = [MetricFilter("duration_seconds", "gte", 10)]
query = dict(content_query=content_query, corpus="both", filters=filters, version=baseline["version"])
selected = client.query_media(**query, limit=20)
query["search_version"] = selected.get("search_version")
print(f"{selected['matching']:,} of {selected['total']:,} videos match; {len(selected['rows'])} shown")
display(pd.DataFrame(selected["rows"]))
with pd.option_context("display.max_rows", None):
    display(metric_coverage(selected))


## Inspect a distribution

These bins cover **all matching videos**, not just the displayed page. Baseline
and filtered counts use the same bin edges. Change the ID to any listed metric.


In [ ]:
metric_id = "duration_seconds"
distribution = selected["distributions"][metric_id]
display(pd.DataFrame(distribution["bins"]))
display(pd.DataFrame([distribution["baseline"], {
    key: distribution[key] for key in ("known", "unknown", "median", "minimum", "maximum")
}], index=["Full dataset", "Filtered"]))


## Complementary-half rehearsal

Each of **5 subjects** receives **16 full videos + 4 first halves + 4 later second-half foils**.
That is 24 presentations from 20 parents, with **12 cut / 12 no-cut actual presentations**.
Paired videos are never shown in full. No parents are shared across subjects.

This recreates the browser demo using the same API; the earlier demo was created in the browser, not by running this notebook.
The server needs the complementary-half allocator, `DBP_CUT_BALANCE_CANDIDATES`, and its pinned `DBP_CUT_BALANCE_SHA256`.
Both rendered halves must have measurements. These local media and metadata are **not included in this repository**.

The recipe below deliberately uses no filters, independent of the exploration above.
The seed reproduces the draw only with unchanged data, candidate measurements, and allocator.
Cut labels are FFmpeg scene-score ≥ 0.3, not manually validated boundaries.
Prior-session exposure and duplicate content are not checked: use fresh rehearsal subjects.


In [ ]:
experiment: Experiment = client.create_experiment(
    name="Complementary halves v3 — 5 subjects × 24 trials",
    seed="cut-demo-24-v3", filters=[], content_query="", corpus="both", media_type=Video,
)
print("Dataset version:", experiment.selection.version)

# Future image example, once supported by the server:
# image_experiment = client.create_experiment(name="Images", seed="images-1", media_type=Image)


## Create or reopen

Set `create_demo = True` to save and publish. Alternatively, copy an ID from the list into
`saved_experiment_id` to reopen without resampling. Creation calls save new records, even with the same seed.
Neither creation nor download records participant presentations.


In [ ]:
display(pd.DataFrame(client.experiments(limit=100)["experiments"]))


In [ ]:
create_demo = False
saved_experiment_id = ""
assignments: Assignments | None = None

if saved_experiment_id:
    assignments = client.assignments(saved_experiment_id)
elif create_demo:
    assignments = experiment.assign(
        subjects=5, items_per_subject=20, blocks=1, foils_per_block=4,
        shared_per_subject=0, repeats_per_subject=0, balance_cuts=True,
    )
    saved_experiment_id = assignments.experiment_id

if assignments is not None:
    print("Save this experiment ID:", assignments.experiment_id)
    print("Subject IDs:", assignments.subject_ids)
else:
    print("Enter a saved experiment ID or enable create_demo, then rerun this cell.")


## Verify the actual presentations

Verify half boundaries, ordering, subject uniqueness, and measured cut balance.
The first-half trial has role `parent` but a nonempty `segment`, so the server delivers only that half.
Missing measurements fail rather than count as no-cut.


In [ ]:
def verify_rehearsal(publication):
    assert publication["foil_policy"] == "complementary-halves-v1", "Wrong foil design"
    assert len(publication["subjects"]) == 5
    rows, all_parents = [], set()
    for subject in publication["subjects"]:
        trials = [trial for block in subject["blocks"] for trial in block["trials"]]
        full = [trial for trial in trials if trial["segment"] is None]
        first = [trial for trial in trials if trial["role"] == "parent" and trial["segment"] is not None]
        foils = [trial for trial in trials if trial["role"] == "foil"]
        assert (len(trials), len(full), len(first), len(foils)) == (24, 16, 4, 4)
        assert all(trial["role"] == "parent" for trial in full)
        parents = {trial["media_id"] for trial in trials}
        assert len(parents) == 20 and all_parents.isdisjoint(parents)
        all_parents.update(parents)
        assert {trial["media_id"] for trial in first} == {trial["media_id"] for trial in foils}
        assert {trial["media_id"] for trial in full}.isdisjoint(trial["media_id"] for trial in foils)
        for foil in foils:
            initial = next(trial for trial in first if trial["media_id"] == foil["media_id"])
            midpoint = initial["segment"]["end_seconds"]
            assert midpoint > 0 and initial["segment"]["start_seconds"] == 0
            assert foil["segment"] == {"start_seconds": midpoint, "end_seconds": 2 * midpoint}
            assert trials.index(initial) < trials.index(foil)
        counts = [trial["cut_measurement"]["hard_cut_count"] for trial in trials]
        assert all(type(count) is int and count >= 0 for count in counts)
        cut, no_cut = sum(count > 0 for count in counts), counts.count(0)
        assert (cut, no_cut) == (12, 12)
        rows.append(dict(subject=subject["subject_id"], full=16, first_halves=4, foils=4, cut=cut, no_cut=no_cut))
    assert len(all_parents) == 100
    return pd.DataFrame(rows)

if assignments is not None:
    publication = client.experiment(assignments.experiment_id)["publication"]
    display(verify_rehearsal(publication))
    print("Verified 100 parents and 120 presentations.")


## Download one subject (optional)

Set `download_demo = True` after verification. Files are already trimmed: do not crop them again.
Downloads verify hashes and require a new destination; reopening a session's workspace restores existing paths.
No media plays and no presentation is recorded.


In [ ]:
download_demo = False
subject_id = "subject-001"

if download_demo:
    if assignments is None:
        raise ValueError("Create or reopen assignments first.")
    verify_rehearsal(client.experiment(assignments.experiment_id)["publication"])
    session: Session = assignments.subject(subject_id, workspace=".dbp")
    destination = Path("dbp-demo-download") / assignments.experiment_id / subject_id
    trials: tuple[Trial, ...] = session.download(destination)
    display(pd.DataFrame([{
        "trial": trial.trial_id, "media": trial.media_id,
        "presentation": "second half (foil)" if trial.role == "foil" else "first half" if trial.segment else "full video",
        "source interval": trial.segment, "path": str(trial.local_path),
    } for trial in trials]))
else:
    print("Download skipped")


## Report presentation from your own task

`Trial` describes the media, block, role, optional foil `Segment`, and local path.
Call `started()` at presentation onset and `completed()` only after successful
playback. If `started()` fails, stop the task. For precise stimulus timing, perform
synchronization outside timing-critical display code; these calls involve network I/O.

Events are saved in a local SQLite journal before upload to the website. IDs make
upload retries safe. `pending_trials` excludes both started and completed trials;
`incomplete_trials` exposes started-but-unfinished trials for review, not automatic
replay. Progress is per trial, so intentional repeats remain separate.
Another device sees only progress that reached the server. Keep the journal after
an interruption and call `sync()` to retry uploads before switching devices.

The loop is commented out so running this notebook cannot falsely record playback.


In [ ]:
# Your presentation framework supplies present_video(), which returns only
# after successful playback and raises if presentation is interrupted.
#
# for trial in session.pending_trials:
#     session.started(trial)
#     present_video(trial.local_path)
#     session.completed(trial)
# session.sync()
#
# Review interrupted trials before continuing an experiment:
# display(session.incomplete_trials)
# print(session.journal_path)
#
# Future image task: use present_image(trial.local_path) instead of present_video.
# Presentation code stays outside dbp_api; progress calls stay the same.


## Disconnect
Run this when finished. Clear outputs before sharing.


In [ ]:
client.logout()
print("Disconnected")
